# Chat Engines & Memory

Every episode so far has used `.query()` — a single, stateless question in, one answer out, with zero awareness of anything asked before it. A **chat engine** adds conversation memory on top of the same index, so follow-up questions like "what about his sister?" resolve correctly using earlier turns as context, the way a real chatbot needs to.


**Step 1 — Set up.** Configures the usual LLM and embedding model that the chat engine will use underneath.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from the HTTP client and LlamaIndex itself.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads .env into os.environ so OPENAI_API_KEY is available to the clients below.
load_dotenv()

Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Build the index and a memory-aware chat engine.** `ChatMemoryBuffer` stores the running conversation, and `chat_mode="condense_plus_context"` uses that history to rewrite each new question into a standalone query before retrieving.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.memory import ChatMemoryBuffer

documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)

memory = ChatMemoryBuffer.from_defaults(token_limit=1500)  # holds up to ~1500 tokens of chat history
# condense_plus_context rewrites each question using chat history before
# retrieving, then answers using both the retrieved nodes and the conversation.
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context", memory=memory)

**Step 3 — Have a multi-turn conversation.** The 2nd and 3rd questions use pronouns ("he", "his sister") that only make sense with the earlier turns as context — exactly what `chat_engine.chat()` uses `memory` for behind the scenes.


In [3]:
turns = [
    "Tell me about Tanjiro Kamado from Demon Slayer.",
    "What breathing technique does he use?",  # "he" only resolves via memory
    "What about his sister?",  # "his sister" also relies on the earlier turn
]

for turn in turns:
    response = chat_engine.chat(turn)  # each call updates `memory` with this turn automatically
    print(f"User: {turn}\nAssistant: {response}\n")
    print("---\n")

User: Tell me about Tanjiro Kamado from Demon Slayer.
Assistant: Tanjiro Kamado is the protagonist of the series Demon Slayer (Kimetsu no Yaiba). He is depicted as a kind-hearted boy who joins the Demon Slayer Corps after his family is slaughtered by demons, and his younger sister Nezuko is turned into a demon. Despite her transformation, Nezuko retains her humanity and fights alongside him. 

Tanjiro trains under the former Hashira Sakonji Urokodaki and initially fights using Water Breathing, a swordsmanship style that mimics the fluid and adaptable properties of water. As the series progresses, he awakens Hinokami Kagura, also known as Sun Breathing, which is the original breathing style from which all other styles descend. This technique is tied to his family's inherited dance ritual.

He is characterized by his empathy and compassion, extending understanding even to demons, viewing many of them as tragic figures rather than pure villains. His primary goal is to find a cure for Nezu

**Step 4 — Inspect what's actually stored in memory.** `memory.get_all()` returns the raw message list the chat engine has been building up — useful for seeing exactly what context future turns have access to.


In [4]:
# memory.get_all() returns every stored ChatMessage — 2 per turn (user + assistant).
print(f"Memory now holds {len(memory.get_all())} messages:\n")
for message in memory.get_all():
    print(f"[{message.role}] {str(message.content)[:80]}")

Memory now holds 6 messages:

[MessageRole.USER] Tell me about Tanjiro Kamado from Demon Slayer.
[MessageRole.ASSISTANT] Tanjiro Kamado is the protagonist of the series Demon Slayer (Kimetsu no Yaiba).
[MessageRole.USER] What breathing technique does he use?
[MessageRole.ASSISTANT] Tanjiro Kamado initially fights using Water Breathing, a swordsmanship style tha
[MessageRole.USER] What about his sister?
[MessageRole.ASSISTANT] Tanjiro's younger sister is Nezuko Kamado. She is a central character in the ser


### Summary

- `chat_mode="condense_plus_context"` rewrites each new question into a standalone query (using chat history) before retrieving, then answers with both the retrieved context and the conversation so far — that's what lets pronouns like "he" and "his sister" resolve correctly.
- `ChatMemoryBuffer` caps memory by token count, not message count — once the limit is hit, the oldest messages drop off first, which matters for long conversations.
